# Laboratorio I y Tarea I Problema de búsqueda

- Inteligencia Artificial | II-2026
- Prof. Marvin Sandí
- Estudiantes: Anthony Sanchez, Andrés Camacho, Enrique Ramírez

El contenido de la tarea se encuentra después de la implementación del `A*`

In [1]:
# importar dependencias
import numpy as np
from collections import deque
import random
import heapq
from itertools import count

In [2]:
# variables globales
COLORES = ('R', 'V', 'A', 'B', 'N')
N_COLUMNAS = 6
N_FILAS = 25 # en realidad es 25 pero luego arreglo esto
N_COLORES = 5 #cantidad de fichas por color
# el objetivo es que cada ficha quede solo con fichas de su mismo color

Puede haber más de un estado objetivo. Por ejemplo este:

In [3]:
# Este estado objetivo se usa para generar un estado aleatorio inicial
# estado_objetivo = np.array([ #ojo esto es preliminar
#     ['V','V','V','V','V'],
#     ['R','R','R','R','R'],
#     ['A','A','A','A','A'],
#     ['-','-','-','-','-'],
#     ['B','B','B','B','B'],
#     ['N','N','N','N','N']

# ])

# todo el código de las lineas siguientes de ESTA celda fue hecho por IA a partir del array comentado arriba
# Se le pidió que ajustara estado_objetivo para que tuviera la forma (25, 6) con las fichas acomodadas al fondo de la columna.
# Se usó IA porque es muy machotero

estado_objetivo = np.full((N_FILAS, N_COLUMNAS), '-')

columnas_colores = ('V', 'R', 'A', '-', 'B', 'N')

for columna, color in enumerate(columnas_colores):
    if color == '-':
        continue
    estado_objetivo[N_FILAS - N_COLORES:, columna] = color

estado_objetivo

array([['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['V', 'R', 'A', '-', 'B', 'N'],
       ['V', 'R', 'A', '-', 'B', 'N'],
       ['V', 'R', 'A', '-', 'B', 'N'],
       ['V', 'R', 'A', '-', 'B', 'N'],
       ['V', 'R', 'A', '-', 'B', 'N']], dtype='<U1')

## Modelo de transición

Una ficha pasa de estar en una columna Ci, a una columna Cj. No se modifica el estado del resto de columnas, siempre y cuando la columna destino esté vacía o tenga en su fila superior una ficha del mismo color que la que se desea colocar.

Asi se representa un estado:

In [4]:
def encontrar_cima(estado, columna): # Buscar color al tope de una columna, si hay

    for fila in range(N_FILAS):
        if estado[fila, columna] == '-': # Si la fila está vacía, sigo a la siguiente
            continue
        return estado[fila, columna], fila # Encontré la primera ficha no vacía, con su fila

    return '-', N_FILAS - 1 # Toda la columna está vacía

In [5]:
def calcular_posibles_transiciones(estado): # para el estado aleatorio, no valida
    transiciones = []

    for origen in range(N_COLUMNAS): # para la cantidad de columnas que hay

        # La siguiente linea de codigo y las ultimas dos de encontrar_cima() las hice con IA
        # Le pedí a la IA que modificara las funciones para que fueran compatibles con la función sucesor()
        ficha, fila_origen = encontrar_cima(estado, origen) # encuentro la primer ficha de esa columna

        for destino in range(N_COLUMNAS):

            # no cuenta la misma columna
            if origen == destino:
                continue

            color_destino, _ = encontrar_cima(estado, destino)

            # # fondo vacío o mismo color
            # if color_destino == '-' or color_destino == ficha:
            transiciones.append((origen, destino, fila_origen))

    return transiciones

In [6]:
def calcular_transiciones_validas(estado): #calcula transiciones y verifica validez
    transiciones = []

    cimas = [encontrar_cima(estado, columna) for columna in range(N_COLUMNAS)]

    for origen in range(N_COLUMNAS): # para la cantidad de columnas que hay

        ficha, fila_origen = cimas[origen]

        for destino in range(N_COLUMNAS):

            # no cuenta la misma columna
            if origen == destino:
                continue

            color_destino, _ = cimas[destino]

            # fondo vacío o mismo color
            if color_destino == '-' or color_destino == ficha:
                transiciones.append((origen, destino, fila_origen))

    return transiciones

In [7]:
def sucesor(matriz, transicion): #  matriz nueva al haber una transicion

    origen, destino, fila_origen = transicion

    matriz_sucesora = matriz.copy()

    # se quita ficha del origen
    ficha = matriz_sucesora[fila_origen, origen]
    matriz_sucesora[fila_origen, origen] = '-'

    # donde cae la ficha en el destino
    color_destino, fila_destino = encontrar_cima(matriz, destino)

    if color_destino == '-':
        fila_libre = fila_destino # cae al fondo
    else:
        fila_libre = fila_destino - 1 # encima del mismo color

    matriz_sucesora[fila_libre, destino] = ficha

    return matriz_sucesora # Se retorna la matriz sucesora.

In [8]:
def generar_estado_aleatorio(estado_objetivo, pasos=40): # generar un estado aleatorio inicial

    # Tomo el estado objetivo
    estado = estado_objetivo.copy()

    # Hago N veces:
        # tomar matriz
        # calcular posibles transiciones
        # tomo una aleatoriamente
        # guardo matriz resultante y repito hasta que pasos llegue a cero
    for _ in range(pasos):

        transiciones = calcular_posibles_transiciones(estado)

        transicion = random.choice(transiciones)

        estado = sucesor(
            estado,
            transicion
        )

    return estado

estado_inicial = generar_estado_aleatorio(estado_objetivo)

estado_inicial

array([['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', '-'],
       ['-', '-', '-', '-', '-', 'R'],
       ['-', '-', 'R', '-', '-', 'B'],
       ['N', '-', 'A', '-', 'B', 'B'],
       ['V', '-', 'R', '-', 'A', 'N'],
       ['V', '-', 'B', '-', 'R', 'N'],
       ['V', 'A', 'A', '-', 'V', 'N'],
       ['V', 'R', 'A', '-', 'B', 'N']], dtype='<U1')

In [9]:
def es_estado_objetivo(estado): # verificar si un estado es uno de los objetivos
    # Se usó IA. Prompt: "podemos crear una mejor forma de validar el 
    # estado objetivo? puede haber mas de un estado objetivo, el que tenemos
    # ahi es de ejemplo para crear un estado aleatorio"

    # Ya el código funcionaba antes de haber hecho esto con IA. 
    columna_de_color = {}

    for columna in range(N_COLUMNAS):

        colores_columna = set(estado[:, columna]) - {'-'}

        # columna no puede tener más de un color
        if len(colores_columna) > 1:
            return False

        if colores_columna:
            color = colores_columna.pop()

            # ese color ya está en otra columna
            if color in columna_de_color and columna_de_color[color] != columna:
                return False

            columna_de_color[color] = columna

    return True

In [10]:
def color_tam_fondo(estado, columna): # cantidad de bloques de un mismo color al fondo de una columna y cuantos son

    fila = N_FILAS - 1
    color = estado[fila, columna]

    if color == '-':
        return None, 0 # no cuenta

    conteo = 0

    while fila >= 0 and estado[fila, columna] == color:
        conteo += 1
        fila -= 1

    return color, conteo


def heuristica(estado): # suma sobre los colores de lo que falta para juntar cada uno

    mejor_bloque_por_color = {color: 0 for color in COLORES}

    for columna in range(N_COLUMNAS):

        color, bloque = color_tam_fondo(estado, columna)

        if color is not None:
            mejor_bloque_por_color[color] = max(mejor_bloque_por_color[color], bloque)

    return sum(N_COLORES - mejor_bloque_por_color[color] for color in COLORES)

## Implementación con algoritmo de búsqueda por anchura

In [11]:
def bfs():

    estados = deque([(estado_inicial, 0)])

    estados_visitados = {
        tuple(estado_inicial.flatten())
    }

    padres = {}

    while estados:

        # Sacar el primer estado de la cola
        matriz_actual, distancia = estados.popleft()

        # ¿Llegamos al objetivo?
        if es_estado_objetivo(matriz_actual):

            print("¡Se halló un estado objetivo!")
            print(f"Se requirieron {distancia} pasos para hallar la solución")

            # Acá se reconstruye la ruta para presentar la forma en que se llegó
            # a la solución objetivo

            solucion = [matriz_actual]
            actual = matriz_actual

            while not np.array_equal(actual, estado_inicial):

                padre = padres[tuple(actual.flatten())]

                solucion.append(padre)

                actual = padre

            # La solución se construyó
            # desde objetivo -> inicial
            solucion.reverse()

            print("\nSOLUCIÓN:")

            for i, estado in enumerate(solucion):

                print(f"\nPaso {i}:")
                print(estado)

            return

        # Calcular movimientos válidos
        transiciones = calcular_transiciones_validas(matriz_actual)

        # Generar sucesores
        for transicion in transiciones:

            nuevo_estado = sucesor(
                matriz_actual,
                transicion
            )

            # Representación para comprobar
            # si ya fue visitado
            estado_nuevo = tuple(
                nuevo_estado.flatten()
            )

            # Evitar estados repetidos
            if estado_nuevo not in estados_visitados:

                estados.append(
                    (nuevo_estado, distancia + 1)
                )

                estados_visitados.add(
                    estado_nuevo
                )

                # Guardar de qué estado proviene
                padres[estado_nuevo] = matriz_actual

    print("No se encontró una solución")


In [12]:
bfs()

¡Se halló un estado objetivo!
Se requirieron 16 pasos para hallar la solución

SOLUCIÓN:

Paso 0:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' 'R']
 ['-' '-' 'R' '-' '-' 'B']
 ['N' '-' 'A' '-' 'B' 'B']
 ['V' '-' 'R' '-' 'A' 'N']
 ['V' '-' 'B' '-' 'R' 'N']
 ['V' 'A' 'A' '-' 'V' 'N']
 ['V' 'R' 'A' '-' 'B' 'N']]

Paso 1:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 

## Implementación con algoritmo `A*`

In [13]:
def a_estrella(): # De esta función, lo que usa heapq fue hecho con IA.
    # Se le dió una versión ya implementada de la función y se dio como prompt:
    # "can we make our a* algorithm faster? somehow?" 

    contador = count() # desempate del heap

    abiertos = [] # estados por explorar, ordenado por f = g + h
    heapq.heappush(abiertos, (heuristica(estado_inicial), 0, next(contador), estado_inicial))

    distancias = {
        tuple(estado_inicial.flatten()): 0
    } # cada estado ya visto y costo para llegar a este

    cerrados = set() # estados que voy sacando

    padres = {} # para cada estado, cual fue el estado anterior

    while abiertos:

        # el heap mantiene en el tope el de menor f = g + h
        f, distancia, _, matriz_actual = heapq.heappop(abiertos)

        # actualizo estado
        estado_actual = tuple(matriz_actual.flatten())

        # ignorar repeticiones
        if estado_actual in cerrados:
            continue

        cerrados.add(estado_actual)

        # ¿Llegamos a un estado objetivo?
        if es_estado_objetivo(matriz_actual):

            print("¡Se halló un estado objetivo!")
            print(f"Se requirieron {distancia} pasos para hallar la solución")

            # Acá se reconstruye la ruta para presentar la forma en que se llegó
            # a la solución objetivo

            solucion = [matriz_actual]
            actual = matriz_actual

            while not np.array_equal(actual, estado_inicial):

                padre = padres[tuple(actual.flatten())]

                solucion.append(padre)

                actual = padre

            # La solución se construyó
            # desde objetivo -> inicial
            solucion.reverse()

            print("\nSOLUCIÓN:")

            for i, estado in enumerate(solucion):

                print(f"\nPaso {i}:")
                print(estado)

            return

        # Calcular movimientos válidos
        transiciones = calcular_transiciones_validas(matriz_actual)

        # Generar sucesores
        for transicion in transiciones:

            nuevo_estado = sucesor(
                matriz_actual,
                transicion
            )

            # para comprobar si ya fue visitado
            estado_nuevo = tuple(
                nuevo_estado.flatten()
            )

            nueva_distancia = distancia + 1

            # Solo se agrega si el camino es mejor que el que ya se tiene
            if estado_nuevo not in distancias or nueva_distancia < distancias[estado_nuevo]:

                distancias[estado_nuevo] = nueva_distancia

                # Guardar de donde proviene
                padres[estado_nuevo] = matriz_actual

                nuevo_f = nueva_distancia + heuristica(nuevo_estado)

                heapq.heappush(abiertos, (nuevo_f, nueva_distancia, next(contador), nuevo_estado))

    print("No se encontró una solución")

In [14]:
a_estrella()

¡Se halló un estado objetivo!
Se requirieron 16 pasos para hallar la solución

SOLUCIÓN:

Paso 0:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' 'R']
 ['-' '-' 'R' '-' '-' 'B']
 ['N' '-' 'A' '-' 'B' 'B']
 ['V' '-' 'R' '-' 'A' 'N']
 ['V' '-' 'B' '-' 'R' 'N']
 ['V' 'A' 'A' '-' 'V' 'N']
 ['V' 'R' 'A' '-' 'B' 'N']]

Paso 1:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 


# Tarea corta: Implementación con algoritmo de IDS*

Basándose en el algoritmo desarrollado durante el laboratorio, deben implementar
uno alternativo que permita encontrar una solución para este juego utilizando IDS*
con las siguientes características:


- Solución recursiva que no haga uso extensivo de memoria.
  
- No debe usar Lista Abierta ni Lista Cerrada sino que debe ir explorando si los nodos ubicados a la profundidad estimada con la heurística ( más la profundidad actual) corresponden al estado final de una solución.  Caso contrario estime la nueva profundidad con el menor valor de heurística calculado sobre los nodos ubicados en la profundidad explorada.

- Debe hacer uso de su heurística para establecer la profundidad de búsqueda de IDS* en cada iteración.
  
Evalúe la complejidad espacial y temporal de los tres algoritmos creados y haga un análisis comparativo.  Para tal efecto debe ejecutar al menos 20 veces cada algoritmo y dar sus resultados basándose en un promedio de las 20 ejecuciones. 

In [15]:
# IDS* Lo explican en la sección 3.53 del libro

# Fuente del pseudocodigo: https://www.youtube.com/watch?v=BUHc8p5Mpdo&t=104s

# IDAAsterisco()
    # paso 1: 
    # root node = estado inicial
    # find the f-score = g(x)+h(x)
    # paso 2:
    # set threshold: maximum f-score allowed for that node for further explorations
    # paso 3:
    # expand current node to its children and find f-scores
    # paso 4:
    # if for any node, f-score > threshold, prune that node because it's too expensive.
    # store node in visited nodes list
    # paso 5:
    # if goal node found, return the path from start node to goal node
    # paso 6:
    # if goal node not found, repeat from step 2 by changing the threshold with the
    # minimum pruned value from the visited node list. Continue algorithm until goal
    # node is reached

def buscar(ruta, g, threshold):

    nodo_actual = ruta[-1]
    f = g + heuristica(nodo_actual) # paso 1

    # paso 4: podar si hace falta
    if f > threshold:
        return f, None

    # paso 5, sí apareció el objetivo
    if es_estado_objetivo(nodo_actual):
        return f, list(ruta)

    minimo_podado = float('inf')

    # paso 3: hijos de nodo actual
    for transicion in calcular_transiciones_validas(nodo_actual):

        hijo = sucesor(nodo_actual, transicion)

        ruta.append(hijo)
        f_podado, solucion = buscar(ruta, g + 1, threshold) #esta es la parte que hice recursiva

        if solucion is not None:
            return f_podado, solucion

        minimo_podado = min(minimo_podado, f_podado) # paso 6

        ruta.pop() # ya se exploró. se quita

    return minimo_podado, None


def IDAAsterisco(estado_inicial):

    threshold = heuristica(estado_inicial) # threshold es f = g(0) + h(inicial)

    while True:

        f_podado, solucion = buscar([estado_inicial], 0, threshold)

        if solucion is not None:

            print("¡Se halló un estado objetivo!")
            print(f"Se requirieron {len(solucion) - 1} pasos para hallar la solución")

            print("\nSOLUCIÓN:")

            for i, estado in enumerate(solucion):
                print(f"\nPaso {i}:")
                print(estado)

            return solucion

        threshold = f_podado # paso 6


In [16]:
IDAAsterisco(estado_inicial)

¡Se halló un estado objetivo!
Se requirieron 16 pasos para hallar la solución

SOLUCIÓN:

Paso 0:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' 'R']
 ['-' '-' 'R' '-' '-' 'B']
 ['N' '-' 'A' '-' 'B' 'B']
 ['V' '-' 'R' '-' 'A' 'N']
 ['V' '-' 'B' '-' 'R' 'N']
 ['V' 'A' 'A' '-' 'V' 'N']
 ['V' 'R' 'A' '-' 'B' 'N']]

Paso 1:
[['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 ['-' '-' '-' '-' '-' '-']
 

[array([['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', '-'],
        ['-', '-', '-', '-', '-', 'R'],
        ['-', '-', 'R', '-', '-', 'B'],
        ['N', '-', 'A', '-', 'B', 'B'],
        ['V', '-', 'R', '-', 'A', 'N'],
        ['V', '-', 'B', '-', 'R', 'N'],
        ['V', 'A', 'A', '-', 'V', 'N'],
        ['V', 'R', 'A', '-', 'B', 'N']],

# Análisis de complejidad temporal y espacial y pruebas

La complejidad temporal de `bfs` es O(b^d*n) porque se explora cada estado en cada "nivel" de profundidad y porque expandir estados es O(n).

La complejidad espacial tambien es O(b^d*n), lo cual hace que sea el algoritmo más lento de los tres. Consume mucha memoria porque se guardan todos los estados.

La complejidad temporal y espacial de `A*` es la misma que la de bfs, pero en realidad es más rápido en la mayoría de los casos porque no se expanden todos los nodos. La heurística hace que funcione más eficientemente.

La complejidad temporal de `IDS*` es también O(b^d*n), pero usa menos memoria que los otros dos, porque no tiene las listas de abierto y cerrado, entonces la complejidad espacial es O(d).




In [ ]:
# funcion para calcular tiempos promedio de corrida para los tres algoritmos que hicimos
import time
import io
import contextlib
 # Odio python
def medir_tiempo():

    tiemposBFS = []
    for pasos in range(0, 20):
        estado_inicial = generar_estado_aleatorio(estado_objetivo, pasos=20)
        inicio = time.perf_counter()
        # La siguiente línea se hizo con IA. Prompt: "how do we disable printing here?"
        with contextlib.redirect_stdout(io.StringIO()):
            bfs()
        fin = time.perf_counter()
        tiempo = fin - inicio
        tiemposBFS.append(tiempo)
        print(f"pasos={pasos:2d} -> tiempo={tiempo:.4f}s")

    promedio = sum(tiemposBFS) / len(tiemposBFS)
    print(f"Promedio (BFS): {promedio:.4f}s\n")


    tiemposAS = []
    for pasos in range(0, 20):
        estado_inicial = generar_estado_aleatorio(estado_objetivo, pasos=20)
        inicio = time.perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):
            a_estrella()
        fin = time.perf_counter()
        tiempo = fin - inicio
        tiemposAS.append(tiempo)
        print(f"pasos={pasos:2d} -> tiempo={tiempo:.4f}s")

    promedio = sum(tiemposAS) / len(tiemposAS)
    print(f"Promedio (A estrella): {promedio:.4f}s\n")


    tiemposIDestrella = []
    for pasos in range(0, 20):
        estado_inicial = generar_estado_aleatorio(estado_objetivo, pasos=20)
        inicio = time.perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):
            IDAAsterisco(estado_inicial)
        fin = time.perf_counter()
        tiempo = fin - inicio
        tiemposIDestrella.append(tiempo)
        print(f"pasos={pasos:2d} -> tiempo={tiempo:.4f}s")

    promedio = sum(tiemposIDestrella) / len(tiemposIDestrella)
    print(f"Promedio (IDA*): {promedio:.4f}s\n")

medir_tiempo()

pasos= 0 -> tiempo=3.0399s
pasos= 1 -> tiempo=3.0183s
pasos= 2 -> tiempo=2.7429s
pasos= 3 -> tiempo=2.7830s
pasos= 4 -> tiempo=3.5815s
pasos= 5 -> tiempo=3.7762s
pasos= 6 -> tiempo=4.5164s
pasos= 7 -> tiempo=4.3527s
pasos= 8 -> tiempo=3.9216s
pasos= 9 -> tiempo=3.7515s
pasos=10 -> tiempo=3.7577s
pasos=11 -> tiempo=3.1602s
pasos=12 -> tiempo=2.9365s
pasos=13 -> tiempo=4.2459s
pasos=14 -> tiempo=3.4853s
pasos=15 -> tiempo=5.3608s
pasos=16 -> tiempo=5.8900s
pasos=17 -> tiempo=4.5060s
pasos=18 -> tiempo=3.4226s
pasos=19 -> tiempo=2.8997s
pasos=20 -> tiempo=3.2550s
pasos=21 -> tiempo=3.3980s
pasos=22 -> tiempo=2.7128s
pasos=23 -> tiempo=3.6336s
pasos=24 -> tiempo=3.4821s
pasos=25 -> tiempo=3.5012s
pasos=26 -> tiempo=4.5999s
pasos=27 -> tiempo=4.3387s
pasos=28 -> tiempo=4.1093s
pasos=29 -> tiempo=4.3695s
pasos=30 -> tiempo=4.4707s
pasos=31 -> tiempo=3.7304s
pasos=32 -> tiempo=5.3800s
pasos=33 -> tiempo=4.1134s
pasos=34 -> tiempo=4.6013s
pasos=35 -> tiempo=5.4236s
pasos=36 -> tiempo=3.6910s
p

# Tabla de resultados de complejidad temporal

| Pasadas | BFS | A* | IDS* |
|---------|------------|------------|------------|
| 20      | 1.2497s    | 0.0649s    | 0.0431s    |
| 30      | 1.7813s    | 0.0673s    | 0.0201s    |
| 40      | 2.6526s    | 0.0807s    | 0.0411s    |

